# Load the legacy loan table

Builds `hermesf_sl_legacy` month by month from `sec_lending_clean_query_legacy.txt`, one statement per month. Months already in the table are skipped, so the notebook can simply be rerun after an interruption. Keep the laptop awake while it runs, the Impala session dies when the connection drops.

In [ ]:
import datetime as dt
import pyodbc
import pandas as pd

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

The select comes from the query file, the date range is replaced per month.

In [ ]:
DB = 'xlab_ecb_prj_sftds_cb_common'
TABLE = f'{DB}.hermesf_sl_legacy'
FULL_RANGE = "BETWEEN '2021-01-01' AND '2026-05-31'"

sql = open('sec_lending_clean_query_legacy.txt').read()
body = sql[sql.index('AS\nSELECT') + 3:].rstrip().rstrip(';')
assert body.count(FULL_RANGE) == 1

months = []
d = dt.date(2021, 1, 1)
while d <= dt.date(2026, 5, 31):
    nxt = (d.replace(day=1) + dt.timedelta(days=32)).replace(day=1)
    months.append((d, min(nxt - dt.timedelta(days=1), dt.date(2026, 5, 31))))
    d = nxt
len(months)

Which months are already in the table?

In [ ]:
cursor.execute(f"SHOW TABLES IN {DB} LIKE 'hermesf_sl_legacy'")
exists = cursor.fetchone() is not None
done = set()
if exists:
    query = f"""
    SELECT SUBSTR(CAST(reference_period AS STRING), 1, 7) AS ym, COUNT(*) AS n
    FROM {TABLE}
    GROUP BY 1
    ORDER BY 1
    """
    df = pd.read_sql_query(query, cnxn)
    done = set(df['ym'])
print(f'{len(done)} / {len(months)} months loaded')

Load the missing months. The first statement creates the table, the following ones append to it.

In [ ]:
failed = []
for start, end in months:
    ym = start.strftime('%Y-%m')
    if ym in done:
        continue
    chunk = body.replace(FULL_RANGE, f"BETWEEN '{start}' AND '{end}'")
    if exists:
        stmt = f"INSERT INTO {TABLE}\n" + chunk
    else:
        stmt = f"""CREATE EXTERNAL TABLE {TABLE}
STORED AS PARQUET TBLPROPERTIES ('external.table.purge'='true')
AS
""" + chunk
    try:
        cursor.execute(stmt)
        exists = True
        done.add(ym)
        cursor.execute(f"SELECT COUNT(*) FROM {TABLE} WHERE reference_period BETWEEN CAST('{start}' AS DATE) AND CAST('{end}' AS DATE)")
        n_month = cursor.fetchone()[0]
        cursor.execute(f"SELECT COUNT(*) FROM {TABLE}")
        n_total = cursor.fetchone()[0]
        print(f"{ym}: inserted {n_month:,} rows, {n_total:,} rows in table ({len(done)} / {len(months)} months)", flush=True)
    except Exception as e:
        failed.append(ym)
        print(f"{ym}: FAILED, {str(e)[:200]}", flush=True)
        cnxn = pyodbc.connect('DSN=Hermes_DSN', autocommit=True)
        cursor = cnxn.cursor()

print('failed months:', failed or 'none')

Rows per month after the run.

In [ ]:
query = f"""
SELECT SUBSTR(CAST(reference_period AS STRING), 1, 7) AS ym, COUNT(*) AS n
FROM {TABLE}
GROUP BY 1
ORDER BY 1
"""
df = pd.read_sql_query(query, cnxn)
df